In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

plt.rcParams['figure.figsize'] = (6, 4)
plt.rcParams['axes.grid'] = True


In [ ]:
g = 9.81
m = 1.0

x0, y0 = 0.0, 1.0
v0 = 20.0
angle_deg = 35.0
angle = np.deg2rad(angle_deg)

vx0 = v0*np.cos(angle)
vy0 = v0*np.sin(angle)

dt = 0.01

t_hit = (vy0 + np.sqrt(vy0**2 + 2*g*y0)) / g
t_end = t_hit
print(f"Analytic flight time t_hit = {t_hit:.4f} s")


In [ ]:
def f(state, t):
    x, y, vx, vy = state
    return np.array([vx, vy, 0.0, -g], dtype=float)

def euler_step(state, t, dt):
    return state + dt*f(state, t)

def rk4_step(state, t, dt):
    k1 = f(state, t)
    k2 = f(state + 0.5*dt*k1, t + 0.5*dt)
    k3 = f(state + 0.5*dt*k2, t + 0.5*dt)
    k4 = f(state + dt*k3, t + dt)
    return state + (dt/6.0)*(k1 + 2*k2 + 2*k3 + k4)

def simulate(method, dt, t_end, x0, y0, vx0, vy0):
    N = int(np.ceil(t_end/dt)) + 1
    t = np.linspace(0.0, dt*(N-1), N)
    X = np.empty(N); Y = np.empty(N); VX = np.empty(N); VY = np.empty(N)
    state = np.array([x0, y0, vx0, vy0], dtype=float)
    X[0], Y[0], VX[0], VY[0] = state
    time = 0.0
    for i in range(1, N):
        state = method(state, time, dt)
        time += dt
        X[i], Y[i], VX[i], VY[i] = state
    return t, X, Y, VX, VY

def analytic_xy(t, x0, y0, vx0, vy0):
    x = x0 + vx0*t
    y = y0 + vy0*t - 0.5*g*t**2
    return x, y

def total_energy(vx, vy, y):
    return 0.5*(vx**2 + vy**2) + g*y


In [ ]:
t_e, X_e, Y_e, VX_e, VY_e = simulate(euler_step, dt, t_end, x0, y0, vx0, vy0)
t_r, X_r, Y_r, VX_r, VY_r = simulate(rk4_step, dt, t_end, x0, y0, vx0, vy0)

Xa_e, Ya_e = analytic_xy(t_e, x0, y0, vx0, vy0)
Xa_r, Ya_r = analytic_xy(t_r, x0, y0, vx0, vy0)

E_e = total_energy(VX_e, VY_e, Y_e)
E_r = total_energy(VX_r, VY_r, Y_r)
E0  = total_energy(vx0, vy0, y0)

err_pos_e = np.sqrt((X_e - Xa_e)**2 + (Y_e - Ya_e)**2)
err_pos_r = np.sqrt((X_r - Xa_r)**2 + (Y_r - Ya_r)**2)
err_E_e = np.abs(E_e - E0)
err_E_r = np.abs(E_r - E0)

print("Base run ready.")

In [ ]:
plt.figure()
plt.plot(Xa_e, Ya_e, label='Аналитика', linewidth=2)
plt.plot(X_e, Y_e, '--', label='Эйлер')
plt.plot(X_r, Y_r, ':', label='RK4')
plt.xlabel('x, м'); plt.ylabel('y, м')
plt.title('Траектория полёта')
plt.legend(); plt.show()

In [ ]:

plt.figure()
plt.plot(t_e, E_e, label='Эйлер')
plt.plot(t_r, E_r, label='RK4')
plt.hlines(E0, 0, t_end, linestyles='dotted', label='E0')
plt.xlabel('t, с'); plt.ylabel('E, Дж (m=1)')
plt.title('Полная энергия')
plt.legend(); plt.show()

plt.figure()
plt.plot(t_e, err_E_e, label='|E - E0|, Эйлер')
plt.plot(t_r, err_E_r, label='|E - E0|, RK4')
plt.yscale('log')
plt.xlabel('t, с'); plt.ylabel('Ошибка энергии')
plt.title('Рост ошибки энергии')
plt.legend(); plt.show()

In [ ]:
plt.figure()
plt.plot(t_e, err_pos_e, label='Эйлер')
plt.plot(t_r, err_pos_r, label='RK4')
plt.yscale('log')
plt.xlabel('t, с'); plt.ylabel('||r_num - r_analyt||')
plt.title('Накопление ошибки координат')
plt.legend(); plt.show()

In [ ]:
angles_deg = np.array([np.pi/12, np.pi/8, np.pi/6, np.pi/4, np.pi/3])
speeds = np.array([10.0, 15.0, 20.0, 25.0])

# фиксированный v0, разные углы
plt.figure()
for ang in angles_deg:
    vx, vy = v0*np.cos(ang), v0*np.sin(ang)
    t_hit_a = (vy + np.sqrt(vy**2 + 2*g*y0)) / g
    tr, Xr, Yr, _, _ = simulate(rk4_step, dt, t_hit_a, x0, y0, vx, vy)
    plt.plot(Xr, Yr, label=f'{np.rad2deg(ang):.0f}°')
plt.xlabel('x, м'); plt.ylabel('y, м')
plt.title(f'Семейство траекторий при v0={v0} м/с')
plt.legend(); plt.show()

# фиксированный угол, разные скорости
fixed_angle_deg = np.pi/6
plt.figure()
for s in speeds:
    vx, vy = s*np.cos(fixed_angle_deg), s*np.sin(fixed_angle_deg)
    t_hit_s = (vy + np.sqrt(vy**2 + 2*g*y0)) / g
    tr, Xr, Yr, _, _ = simulate(rk4_step, dt, t_hit_s, x0, y0, vx, vy)
    plt.plot(Xr, Yr, label=f'v0={s:.0f} м/с')
plt.xlabel('x, м'); plt.ylabel('y, м')
plt.title(f'Семейство при угле {np.rad2deg(fixed_angle_deg):.0f}°')
plt.legend(); plt.show()

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import matplotlib as mpl

mpl.rcParams['animation.html'] = 'jshtml'  # один раз на сессию если в google collab

angles_anim_deg = np.array([np.pi/12, np.pi/8, np.pi/6, np.pi/4, np.pi/3])
traj = []
t_max = 0.0
for ang in angles_anim_deg:
    vx, vy = v0*np.cos(ang), v0*np.sin(ang)
    t_hit_a = (vy + np.sqrt(vy**2 + 2*g*y0)) / g
    t_arr, X, Y, _, _ = simulate(rk4_step, dt, t_hit_a, x0, y0, vx, vy)
    traj.append((t_arr, X, Y))
    t_max = max(t_max, t_arr[-1])

T = np.arange(0.0, t_max+dt, dt)
Xs = np.zeros((len(traj), len(T)))
Ys = np.zeros((len(traj), len(T)))
for i, (t_arr, X, Y) in enumerate(traj):
    n = len(t_arr)
    Xs[i, :n] = X
    Ys[i, :n] = Y
    if n < len(T):
        Xs[i, n:] = X[-1]
        Ys[i, n:] = Y[-1]

fig, ax = plt.subplots()
ax.set_title('Анимация полёта: несколько углов')
ax.set_xlabel('x, м'); ax.set_ylabel('y, м')
ax.set_xlim(0, Xs.max()*1.05)
ax.set_ylim(0, max(Ys.max(), y0)*1.1)

lines = [ax.plot([], [], label=f'{round(np.rad2deg(a), 5)}°')[0] for a in angles_anim_deg]
points = [ax.plot([], [], 'o')[0] for _ in angles_anim_deg]
ax.legend()

def init():
    for ln, pt in zip(lines, points):
        ln.set_data([], [])
        pt.set_data([], [])
    return tuple(lines + points)

def update(frame):
    for i, (ln, pt) in enumerate(zip(lines, points)):
        ln.set_data(Xs[i, :frame+1], Ys[i, :frame+1])
        x = Xs[i, frame]
        y = Ys[i, frame]
        pt.set_data([x], [y])   # <= список, не скаляр
    return tuple(lines + points)

#если не в google collab
#ani = FuncAnimation(fig, update, frames=len(T), init_func=init, interval=20, blit=True)
#plt.show()

#если в google collab
mpl.rcParams['animation.html'] = 'jshtml'  # один раз на сессию

ani = FuncAnimation(fig, update, frames=len(T), init_func=init, interval=20, blit=True)

plt.close(fig)          # чтобы не показать пустой рисунок
HTML(ani.to_jshtml())   # либо: HTML(ani.to_html5_video())

In [ ]:
def range_analytic(theta, v0, h):
    s, c = np.sin(theta), np.cos(theta)
    return (v0*c/g) * (v0*s + np.sqrt((v0*s)**2 + 2*g*h))

def simulate_range_numeric(theta, v0, h, dt):
    vx, vy = v0*np.cos(theta), v0*np.sin(theta)
    t_hit_est = (vy + np.sqrt(vy**2 + 2*g*h)) / g
    t, X, Y, VX, VY = simulate(rk4_step, dt, t_hit_est, 0.0, h, vx, vy)
    idx = np.where(Y >= 0)[0]
    if len(idx) == 0:
        return 0.0
    k = idx[-1]
    if k == len(Y)-1:
        return X[-1]
    y1, y2 = Y[k], Y[k+1]
    x1, x2 = X[k], X[k+1]
    if y2 == y1:
        return X[k]
    alpha = y1 / (y1 - y2)
    return x1 + alpha*(x2 - x1)

thetas = np.deg2rad(np.linspace(1.0, 89.0, 2000))
R_an = range_analytic(thetas, v0, y0)
R_num = np.array([simulate_range_numeric(th, v0, y0, dt) for th in thetas])

i_an = np.argmax(R_an); i_num = np.argmax(R_num)
theta_an_opt = thetas[i_an]; theta_num_opt = thetas[i_num]
R_an_opt = R_an[i_an]; R_num_opt = R_num[i_num]

print(f'Аналитика: θ* ≈ {np.rad2deg(theta_an_opt):.3f}°, R* ≈ {R_an_opt:.3f} м')
print(f'Численно:  θ* ≈ {np.rad2deg(theta_num_opt):.3f}°, R* ≈ {R_num_opt:.3f} м')
print(f'Δθ ≈ {abs(np.rad2deg(theta_an_opt-theta_num_opt)):.4f}°')

plt.figure()
plt.plot(np.rad2deg(thetas), R_an, label='R(θ) аналит.')
plt.plot(np.rad2deg(thetas), R_num, '--', label='R(θ) числ. (RK4)')
plt.axvline(np.rad2deg(theta_an_opt), linestyle=':', label='θ* аналит.')
plt.axvline(np.rad2deg(theta_num_opt), linestyle='-.', label='θ* числ.')
plt.xlabel('θ, °'); plt.ylabel('R, м')
plt.title(f'Дальность vs угол при v0={v0} м/с, h={y0} м')
plt.legend(); plt.show()